#K-mer Spectrum Analysis

K-mer spectrum analysis is a powerful approach for characterizing a genome before assembly. By examining the frequency distribution of short subsequences (k-mers) in raw sequencing reads, we can infer key biological and technical properties of the dataset. These include estimated genome size, levels of heterozygosity, repetitive content, and potential sequencing or contamination issues.

In this notebook, we will walk through a complete workflow using Phytophthora cinnamomi Illumina reads. The steps include:

Downloading raw reads from the SRA

Assessing read quality with FastQC

Trimming low-quality bases and adapters

Counting k-mers with Jellyfish

Analyzing the k-mer histogram with GenomeScope

Together, these analyses provide a foundational understanding of the genome and guide decisions for downstream assembly and annotation.

Install Required Tools and Dependencies

In [ ]:
!apt-get update
!apt-get install -y jellyfish

In [ ]:
!apt-get update -qq
!apt-get install -y -qq sra-toolkit

In [ ]:
!apt-get update -qq
!apt-get install -y fastqc

Download Phytophthora cinnamomi Illumina Reads from SRA

We will work with the Phytophthora cinnamomi Illumina dataset SRR26197585 available in the NCBI SRA database. Use the SRA Toolkit to download the raw sequencing reads and prepare them for quality control and downstream analyses.

https://www.ncbi.nlm.nih.gov/sra/?term=SRR26197585

In [ ]:
!fasterq-dump SRR26197585	 -e 4 -p

Run FastqC for your reads

In [ ]:
!fastqc *.fastq

Review FastQC Results and Decide on Trimming

Examine the FastQC reports to assess read quality, adapter contamination, and sequence composition. Based on these metrics, decide on the appropriate trimming strategy before performing the k-mer spectral analysis. Proper trimming ensures accurate genome size estimation and reduces noise in downstream analyses.

Install Trim_galore

In [ ]:
%%bash
# 1️⃣  Install micromamba (lightweight conda)
wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba > /dev/null

# 2️⃣  Initialize the shell
eval "$(./bin/micromamba shell hook -s bash)"

# 3️⃣  Create an isolated environment (Python 3.12 avoids cutadapt conflict)
micromamba create -y -n tg -c conda-forge -c bioconda python=3.12 trim-galore cutadapt fastqc

# 4️⃣  Confirm installation
micromamba run -n tg trim_galore --version
micromamba run -n tg cutadapt --version

Trim Low-Quality Bases from Paired-End Reads

Based on the quality assessment, we will trim the first 15 bases of each read pair and remove the last 10 bases from both reads. A default Phred quality cutoff of 20 will also be applied to ensure high-quality data for downstream analyses.

In [ ]:
%%bash
# Activate micromamba shell
eval "$(./bin/micromamba shell hook -s bash)"
micromamba run -n tg trim_galore --paired --clip_R1 15 --clip_R2 15 --three_prime_clip_R1 10 --three_prime_clip_R2 10 --fastqc SRR26197585_1.fastq SRR26197585_2.fastq


Now that we have high-quality filtered reads, we can use Jellyfish to compute the k-mer spectrum. This step counts all k-mers in the dataset and produces a histogram that will be used for downstream genome size and complexity estimation.

Begin by running Jellyfish.

In [ ]:
!jellyfish count -C -m 21 -s 1000000000 -t 10 SRR26197585_1_val_1.fq SRR26197585_2_val_2.fq -o reads.jf

Generate a histogram that visualizes the distribution of k-mer frequencies in your sequencing reads.

In [ ]:
!jellyfish histo reads.jf > reads.histo

Take a look at the k-mer histogram file. This distribution will help you evaluate sequencing quality, detect potential contamination, and understand genome characteristics such as heterozygosity and repeat content.

In [ ]:
!head reads.histo

GenomeScope Analysis

Download the k-mer histogram file reads.histo generated from your sequencing data.

Upload this file to the GenomeScope 2.0 web server.
http://genomescope.org/genomescope2.0/

Run the k-mer spectrum analysis to estimate:

Genome size

Haploid genome length

Heterozygosity

Repeat content

Discuss the results in the context of your organism’s biology and sequencing strategy.